# 教程 4：三种循环 —— 层循环、步循环、会话循环

"循环"不是框架写死的控制流，而是三种**作用域**不同的机制。本篇各做一个：

| 循环 | 一句话 | 机制 |
|------|--------|------|
| 层循环 | 更新发布后，同一 decode 步内把 5–8 层**重跑一遍** | `ConditionalLayerReplay((5, 8))` |
| 步循环 | 每一步踩在上一步的终态上 | `Recurrence`（上一步源层 hidden 注入目标层） |
| 会话循环 | 新请求带着上一请求的终态重新生成 | route 共享 lineage + `Capture` 写 memory |

先拿一个无 workload 的基线文本作对照。

In [1]:
import os
os.environ.setdefault("VLLM_ENABLE_V1_MULTIPROCESSING", "0")

import gc
from dataclasses import dataclass
from typing import Annotated

import torch
import pllm
from pllm import TensorUpdate
from vllm import SamplingParams
from pllm.config import VllmDType, ModelConfig
from pllm.contracts.parameter_overlay import ParameterModuleKind, ParameterRole

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
PROMPT = "The sky is"

def config(max_num_seqs=1):
    return ModelConfig(MODEL, VllmDType.BFLOAT16, 64, max_num_seqs, 64, 0.2, seed=7)

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()

def text_of(handle, prompt, **kwargs):
    return handle.generate([prompt], SamplingParams(max_tokens=16,
        temperature=0.0, ignore_eos=True), use_tqdm=False, **kwargs)[0].outputs[0].text

plain = pllm.make_llm(config=config())
baseline = text_of(plain, PROMPT)
plain.llm_engine.engine_core.shutdown()
del plain
cleanup()
print("baseline:", baseline)

/workspace/pllm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-10 12:41:57,961	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 09-10 12:41:58 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 64, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 64, 'max_num_seqs': 1, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 12:41:59 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 12:42:00 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 12:42:00 [model.py:1883] Using max model len 64


... (25 outputs left out)

INFO 09-10 12:42:08 [jit_monitor.py:79] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.


INFO 09-10 12:42:08 [core.py:355] init engine (profile, create kv cache, warmup model) took 1.64 s


INFO 09-10 12:42:12 [hf.py:540] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


baseline:  a beautiful place. It is a place of endless possibilities. The sky is a


## 层循环：更新当步生效，而不是下一步

回忆教程 3 的 overlay：更新在 `MODEL_UPDATE` 时机发布。默认情况下，发布的
新参数从**下一步**起才被模型看到。给 overlay 加一句
`replay=pllm.ConditionalLayerReplay((5, 8))`，发布后 5–8 层在**同一步内**
按签名层段重跑 —— 更新当场生效。

签名约束只有一条：replay 层必须全部属于 overlay 的 targets。
下面的两个 workload 更新数学完全相同，只差这一句 replay。

In [2]:
TARGETS = tuple(pllm.ParameterTarget(layer, ParameterModuleKind.MLP,
    ParameterRole.DOWN_PROJECTION_WEIGHT) for layer in (5, 6, 7, 8))
LR = 0.05
RANK = 2


class LayerLoop(pllm.Workload):
    """5-8 层的低秩 overlay；每次发布后按签名层段重跑。"""

    factors = pllm.RequestState({
        "a": pllm.Buffer(lambda inputs, outputs: (RANK, inputs)),
        "b": pllm.Buffer(lambda inputs, outputs: (outputs, RANK)),
    }, dtype=torch.bfloat16)

    @pllm.on(pllm.Phase.MODEL_UPDATE)
    @pllm.overlay(state="factors", targets=TARGETS,
                  operator=pllm.ParameterOverlayOperator.LOW_RANK_ADD,
                  factors=("a", "b"), scale=1.0, decode=True,
                  replay=pllm.ConditionalLayerReplay((5, 8)))
    def update(self, x: Annotated[torch.Tensor, pllm.Operand.INPUT],
               y: Annotated[torch.Tensor, pllm.Operand.TARGET],
               base: Annotated[torch.Tensor, pllm.Operand.WEIGHT],
               a: Annotated[torch.Tensor, pllm.Parameter("factors", "a")],
               b: Annotated[torch.Tensor, pllm.Parameter("factors", "b")]) -> TensorUpdate:
        output = x @ (base + b @ a).T
        a.add_((LR * y[:, :RANK].T @ x).to(a))
        b.add_((LR * (y.T @ y)[:, :RANK]).to(b))
        return TensorUpdate(output, changed=True, update_count=1, trained_rows=x.shape[0])


class NoReplay(pllm.Workload):
    """同样的更新，不声明 replay —— 更新从下一步起生效。"""

    factors = pllm.RequestState({
        "a": pllm.Buffer(lambda inputs, outputs: (RANK, inputs)),
        "b": pllm.Buffer(lambda inputs, outputs: (outputs, RANK)),
    }, dtype=torch.bfloat16)

    @pllm.on(pllm.Phase.MODEL_UPDATE)
    @pllm.overlay(state="factors", targets=TARGETS,
                  operator=pllm.ParameterOverlayOperator.LOW_RANK_ADD,
                  factors=("a", "b"), scale=1.0, decode=True)
    def update(self, x: Annotated[torch.Tensor, pllm.Operand.INPUT],
               y: Annotated[torch.Tensor, pllm.Operand.TARGET],
               base: Annotated[torch.Tensor, pllm.Operand.WEIGHT],
               a: Annotated[torch.Tensor, pllm.Parameter("factors", "a")],
               b: Annotated[torch.Tensor, pllm.Parameter("factors", "b")]) -> TensorUpdate:
        output = x @ (base + b @ a).T
        a.add_((LR * y[:, :RANK].T @ x).to(a))
        b.add_((LR * (y.T @ y)[:, :RANK]).to(b))
        return TensorUpdate(output, changed=True, update_count=1, trained_rows=x.shape[0])


loop = pllm.make_llm(workload=LayerLoop(), config=config())
replayed = text_of(loop, PROMPT)
loop.close()
cleanup()
noreplay = pllm.make_llm(workload=NoReplay(), config=config())
stepped = text_of(noreplay, PROMPT)
noreplay.close()
cleanup()

print("replay:   ", replayed)
print("no replay:", stepped)
assert replayed != stepped, "同一步内重跑改变了输出轨迹"

INFO 09-10 12:42:13 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 64, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 64, 'max_num_seqs': 1, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_cls': 'pllm.worker.GPUWorker', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 12:42:13 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 12:42:14 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 12:42:14 [model.py:1883] Using max model len 64


INFO 09-10 12:42:14 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])


INFO 09-10 12:42:15 [core.py:121] Initializing a V1 LLM engine (v0.27.1) with config: model='Qwen/Qwen2.5-0.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-0.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=64, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collec

... (24 outputs left out)

INFO 09-10 12:42:22 [model_runner.py:329] Model loading took 0.93 GiB and 1.048238 seconds


INFO 09-10 12:42:23 [gpu_worker.py:789] Free memory on device (11.63/23.48 GiB) on startup. Desired GPU memory utilization is (0.2, 4.7 GiB). Actual usage is 0.98 GiB for consumed memory (weights + non-torch), 0.0 GiB for peak activation, and 0.0 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=3834869760` (3.57 GiB) to fit into requested memory, or `--kv-cache-memory=11282905088` (10.51 GiB) to fully utilize gpu memory. Current kv cache memory in use is 3.72 GiB.


INFO 09-10 12:42:23 [core.py:355] init engine (profile, create kv cache, warmup model) took 0.89 s


replay:     a blueseeing恤恤 '..', '..', '..',יהןיהןיהןיהןיהןיהןיהןיהן
no replay:  a blueseeing恤恤 '..',恤 '..',יהןיהןיהןיהןיהןיהןיהןיהן


## 步循环：上一步的终态注入这一步

`Recurrence(source_layer=-1, destination_layer=0, alpha=1.0)`：每一步把**上一步**
最后一层的 hidden 注入第 0 层。整个 workload 就是一个 `Recurrence` 成员 ——
没有任何任务代码；层号、混合权重、ramp 全部进签名。

In [3]:
@dataclass(frozen=True, slots=True)
class StepLoop(pllm.Workload):
    name = "step_loop"
    config: pllm.Recurrence = pllm.Recurrence(
        source_layer=-1, destination_layer=0, alpha=1.0, beta=None, ramp_tokens=0)


step = pllm.make_llm(workload=StepLoop(), config=config())
recirculated = text_of(step, PROMPT)
step.close()
cleanup()
print("step loop:", recirculated)
assert recirculated != baseline

INFO 09-10 12:42:24 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 64, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 64, 'max_num_seqs': 1, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_cls': 'pllm.worker.GPUWorker', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 12:42:25 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 12:42:25 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 12:42:25 [model.py:1883] Using max model len 64


INFO 09-10 12:42:25 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])


INFO 09-10 12:42:27 [core.py:121] Initializing a V1 LLM engine (v0.27.1) with config: model='Qwen/Qwen2.5-0.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-0.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=64, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collec

... (6 outputs left out)

INFO 09-10 12:42:28 [model_runner.py:329] Model loading took 0.93 GiB and 1.071638 seconds


INFO 09-10 12:42:29 [gpu_worker.py:789] Free memory on device (11.63/23.48 GiB) on startup. Desired GPU memory utilization is (0.2, 4.7 GiB). Actual usage is 0.98 GiB for consumed memory (weights + non-torch), 0.0 GiB for peak activation, and 0.0 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=3834869760` (3.57 GiB) to fit into requested memory, or `--kv-cache-memory=11282905088` (10.51 GiB) to fully utilize gpu memory. Current kv cache memory in use is 3.72 GiB.


INFO 09-10 12:42:29 [core.py:355] init engine (profile, create kv cache, warmup model) took 0.90 s


step loop:  a new study that shows that the use of a new drug called T-2


## 会话循环：带着上一个会话的终态重新生成

教程 3 里 route 键让请求共享 lineage。再加一块积木：**memory 字段与捕获**。

- `memory={"carry": ...}` 声明一个不参与优化的普通字段，它跟随训练发布；
- 训练输入上的第二个标注 `pllm.Capture("session", "carry")` 表示：发布时把
  这个输入的**最后一行**写进 `carry`；
- memory 的读取是**输入版本**语义：这一步读到的是上一步发布的值。

于是：请求 A 的每一步都把当前 hidden 写进 `carry`；A 结束后 `carry` 里就是
**A 的终态 hidden**；同 route 的请求 B 第一步就能读到它。

怎么证明 B 真的读到了？训练损失依赖 `carry`（`prediction = h @ proj.T + carry`），
所以对同一 prompt 比较首步梯度：A（零 carry）≠ B（A 的终态）≠ ……而另一个
全新 route C 的梯度应当与 A **完全一致**（零状态确定性）。

打开 trace 的梯度选择器就能把这些梯度拿出来看。

In [4]:
class SessionLoop(pllm.Workload):
    """route 共享的会话状态：proj 参与训练，carry 是跨请求的 memory 字段。"""

    session = pllm.RequestState(
        {"proj": lambda m: (m.hidden_size, m.hidden_size)},
        optimizer=pllm.AdamW(lr=1e-3),
        memory={"carry": lambda m: (m.hidden_size,)},
        keys=(pllm.RequestKeyField.ROUTE,))

    def ready(self, versions: Annotated[torch.Tensor, pllm.StateVersion("session")]) -> torch.Tensor:
        return versions > 0

    @pllm.on(pllm.Phase.BEFORE_SAMPLE, rows="ready")
    @pllm.guide_sampling(beta=0.2)
    def predict(self, h: Annotated[torch.Tensor, pllm.Hidden(-1)],
                proj: Annotated[torch.Tensor, pllm.Parameter("session", "proj")],
                carry: Annotated[torch.Tensor, pllm.Parameter("session", "carry")],
                versions: Annotated[torch.Tensor, pllm.StateVersion("session")]) -> torch.Tensor:
        return torch.bmm(h.unsqueeze(1), proj).squeeze(1) + carry

    @pllm.on(pllm.Phase.AFTER_SAMPLE)
    @pllm.train(state="session", reduction="state_mean")
    def learn(self,
              h: Annotated[torch.Tensor, pllm.Hidden(-1), pllm.Capture("session", "carry")],
              proj: Annotated[torch.Tensor, pllm.Parameter("session", "proj")],
              carry: Annotated[torch.Tensor, pllm.Parameter("session", "carry")],
              groups: Annotated[torch.Tensor, pllm.StateIndex("session")]) -> torch.Tensor:
        prediction = torch.bmm(h.unsqueeze(1), proj).squeeze(1) + carry
        losses = (prediction - h).square().mean(1)
        counts = torch.bincount(groups).to(losses)
        return losses.new_zeros(counts.shape).index_add(0, groups, losses) / counts


# route lineage 在请求结束后休眠保留，需要额外的 lineage 容量
session = pllm.make_llm(workload=SessionLoop(), config=config(max_num_seqs=2))

INFO 09-10 12:42:30 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 64, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 64, 'max_num_seqs': 2, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_cls': 'pllm.worker.GPUWorker', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 12:42:30 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 12:42:31 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 12:42:31 [model.py:1883] Using max model len 64


INFO 09-10 12:42:31 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])


INFO 09-10 12:42:33 [core.py:121] Initializing a V1 LLM engine (v0.27.1) with config: model='Qwen/Qwen2.5-0.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-0.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=64, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collec

... (8 outputs left out)

WARNING 09-10 12:42:35 [jit_monitor.py:135] Triton kernel JIT compilation during inference: _prepare_prefill_inputs_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


WARNING 09-10 12:42:35 [jit_monitor.py:135] Triton kernel JIT compilation during inference: _prepare_pos_seq_lens_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


WARNING 09-10 12:42:35 [jit_monitor.py:135] Triton kernel JIT compilation during inference: _gather_block_tables_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


INFO 09-10 12:42:35 [core.py:355] init engine (profile, create kv cache, warmup model) took 0.91 s


In [5]:
from pllm.execution.trace import TraceSelection, TracePhase

graph = session.attachment.inputs.plan.semantic_graph
learn_index = next(i for i, task in enumerate(graph.tasks) if str(task.symbol) == "learn")
learn_id = next(task.task_id for task in session.attachment.inputs.plan.schedule_physical.tasks
                if task.task_id == learn_index)


def first_gradient(prompt, route):
    """该次 generate 新增的首个 learn 梯度（trace 是会话级有界缓冲，用游标切片）。"""
    before = session.trace_snapshot()
    cursor = before.dropped + len(before.events)
    session.observe(TraceSelection(tasks=(learn_id,), gradients=True, max_tensor_bytes=32768))
    try:
        session.generate([prompt], SamplingParams(max_tokens=4, temperature=0.0,
            ignore_eos=True), use_tqdm=False, routes=(route,))
        snapshot = session.trace_snapshot()
        events = [e for e in snapshot.events[max(0, cursor - snapshot.dropped):]
                  if e.phase is TracePhase.GRADIENT]
        return events[0].tensor() if events else None
    finally:
        session.observe(TraceSelection())


grad_a = first_gradient(PROMPT, "s")      # 新 lineage：carry 为零
grad_b = first_gradient(PROMPT, "s")      # 加入 A 的 lineage：carry = A 的终态 hidden
grad_c = first_gradient(PROMPT, "fresh")  # 新 route：又是零

print("首步梯度范数 A/B/C:",
      grad_a.norm().item(), grad_b.norm().item(), grad_c.norm().item())
assert not torch.allclose(grad_a, grad_b), "同一路由的重新生成读到了存储的终态"
assert torch.allclose(grad_a, grad_c), "新路由从零 carry 开始"
session.close()

WARNING 09-10 12:42:35 [jit_monitor.py:135] Triton kernel JIT compilation during inference: _bmm_outer_product_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


首步梯度范数 A/B/C: 0.3170323371887207 0.16873608529567719 0.3170323371887207


## 要点回顾

- **层循环** = overlay + `ConditionalLayerReplay(层段)`；replay 层必须是 targets 的子集。
- **步循环** = 一个 `Recurrence` 成员，零任务代码。
- **会话循环** = route 共享 lineage + `Capture` 写 memory 字段；跨请求传递的是
  **状态版本**，不引入任何请求侧历史对象。
- trace 游标 `dropped + len(events)` 同样适用于跨多次 generate 的梯度/损失观察。

下一篇：[教程 5：decode 时在线训练](05_decode_training.ipynb) —— 跨步与跨请求的训练目标。